In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
import importlib
from pathlib import Path
from torch.utils.data import DataLoader
import paths
import build_caption_index as build_caption_index_module

# Reload module de notebook dung code moi nhat.
importlib.reload(paths)
importlib.reload(build_caption_index_module)

from solve_kis import (
    solve_kis,
    multi_query_search,
    reciprocal_rank_fusion,
    normalize_scores,
    static_weight_fusion_3_way,
    finalize_vlm_ranking

)

from siglip_encoder import SiglipEncoder
from search_clip import SiglipRetriever 
from caption_retriever import CaptionRetriever
from paths import (
    KIS_ROOT,
    DATA_DIR,
    MAPPING_DIR,
    CAPTION_DIR,
    CAPTION_MAPPING_DIR,
    INDEX_DIR,
    video_caption_path,
    stored_path,
 )


c:\Users\Khanh\miniconda3\envs\focusguard_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from rank_bm25 import BM25Okapi
import numpy as np

class BM25Retriever:
    def __init__(self, caption_data):
        """
        Nhận vào danh sách caption_data đã được nạp từ các file JSON.
        Định dạng mỗi phần tử: {"video_id": "...", "frame_id": 123, "retrieval_text": "..."}
        """
        self.caption_data = caption_data
        print(f"[*] Khởi tạo BM25 cho {len(caption_data)} frames...")
        
        # Tách từ đơn giản (Tokenization) bằng khoảng trắng và chuyển chữ thường
        self.corpus_tokens = [
            str(item.get("retrieval_text", "")).lower().split() 
            for item in self.caption_data
        ]
        
        # Khởi tạo thuật toán BM25
        self.bm25 = BM25Okapi(self.corpus_tokens)
        print("[*] BM25 Indexing hoàn tất!")

    def search_bm25(self, query, top_k=100):
        # Tách từ câu truy vấn
        query_tokens = query.lower().split()
        
        # Lấy điểm số BM25 cho toàn bộ tập dữ liệu
        scores = self.bm25.get_scores(query_tokens)
        
        # Lấy index của top_k điểm cao nhất
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            score = float(scores[idx])
            if score > 0: # Chỉ lấy những frame có ít nhất 1 từ khóa trùng khớp
                item = self.caption_data[idx]
                results.append({
                    "video_id": item["video_id"],
                    "frame_id": item["frame_id"],
                    "score": score,
                    "retrieval_text": item.get("retrieval_text", "")
                })
                
        return results

In [4]:
encoder = SiglipEncoder()
retriever = SiglipRetriever()

print("SigLIP OK")

[*] Đang nạp mô hình SigLIP trên cpu...


[*] Nạp SigLIP hoàn tất.
Connected to Milvus: clip_keyframes
SigLIP OK


In [5]:
print("[*] Đang khởi tạo Caption Retriever (toàn bộ dữ liệu)...")

try:
    caption_retriever = CaptionRetriever()
    
    # Thống kê nhanh số video có trong mapping để dễ kiểm tra dữ liệu.
    unique_videos = sorted(
        {item["video_id"] for item in caption_retriever.mapping}
    ) if caption_retriever.mapping else []

    print(
        f"[*] caption_retriever sẵn sàng: "
        f"{caption_retriever.index.ntotal} vectors, "
        f"{len(unique_videos)} videos"
    )

except FileNotFoundError as e:
    caption_retriever = None
    print(f"[Lỗi] Không tìm thấy dữ liệu caption/index/mapping: {e}")
    print(
        "    Hãy đảm bảo đã có file trong các thư mục:\n"
        "    - caption_generator/Lxx/*.json\n"
        "    - index/Lxx/*.index\n"
        "    - caption_mapping/Lxx/*.json"
    )

[*] Đang khởi tạo Caption Retriever (toàn bộ dữ liệu)...
[Caption] Loading SentenceTransformer...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2811.98it/s]


[Caption] Loaded 160793 caption vectors
[*] caption_retriever sẵn sàng: 160793 vectors, 774 videos


In [6]:
# ==============================================================================
# CELL NOTEBOOK: LẤY DATA CAPTION TỪ CAPTION_RETRIEVER VÀ CHẠY BM25
# ==============================================================================

all_captions = []

# 1. Kiểm tra nếu caption_retriever chứa danh sách caption_data trực tiếp[cite: 4]
if hasattr(caption_retriever, 'caption_data') and isinstance(caption_retriever.caption_data, list):
    all_captions = caption_retriever.caption_data

# 2. Kiểm tra nếu caption_retriever chứa từ điển captions_db[cite: 12]
elif hasattr(caption_retriever, 'captions_db') and isinstance(caption_retriever.captions_db, dict):
    all_captions = list(caption_retriever.captions_db.values())

# 3. Kiểm tra nếu là MultiVideoCaptionRetriever chứa từ điển retrievers[cite: 11, 12]
elif hasattr(caption_retriever, 'retrievers') and isinstance(caption_retriever.retrievers, dict):
    for vid, retriever in caption_retriever.retrievers.items():
        if hasattr(retriever, 'caption_data'):
            all_captions.extend(retriever.caption_data)
        elif hasattr(retriever, 'captions_db'):
            all_captions.extend(list(retriever.captions_db.values()))

print(f"[*] Đã thu thập được {len(all_captions)} frames caption để nạp vào BM25.")



[*] Đã thu thập được 160793 frames caption để nạp vào BM25.


In [7]:
from object_filter import ObjectMetadataLookup
from apply_object_filter import apply_object_filter
object_lookup = ObjectMetadataLookup(
    data_dir=DATA_DIR
)

print("Object Lookup OK")

Object Lookup OK


In [8]:
bm25_engine = BM25Retriever(all_captions)

[*] Khởi tạo BM25 cho 160793 frames...
[*] BM25 Indexing hoàn tất!


In [9]:
from vlm_reranker import VLMReranker
from paths import video_image_dir
import os

# Khởi tạo mô hình chấm điểm (Quá trình này tốn vài phút để tải checkpoint)
print("[*] Đang nạp Qwen2.5-VL Reranker lên GPU...")
vlm_engine = VLMReranker()
print("[*] VLM đã sẵn sàng!")

[*] Đang nạp Qwen2.5-VL Reranker lên GPU...
[VLM] Loading model...
[VLM] Device: cuda:0
[VLM] GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
[VLM] Total VRAM: 6.4 GB


Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 449.98it/s]
c:\Users\Khanh\miniconda3\envs\focusguard_ai\Lib\site-packages\transformers\modeling_utils.py:5090: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)
Loading weights: 100%|██████████| 824/824 [00:20<00:00, 41.19it/s]


[VLM] Model loaded
[*] VLM đã sẵn sàng!


In [119]:
import json

with open(KIS_ROOT / "test.json", "r", encoding="utf-8") as f:
    structured_query = json.load(f)

print(json.dumps(structured_query, ensure_ascii=False, indent=2))

{
  "query_id": "q_057",
  "raw_query": "Camera lia chậm trên mặt bàn làm việc, bắt đầu từ một bức ảnh/tranh một người đang bắt tay với Bác Hồ, sau đó chuyển sang phần bên phải là tay nghệ nhân đang vẽ lại chân dung người bắt tay Bác. Người nghệ nhân này vẽ bằng một cây bút đặc biệt trên một chất liệu đặc biệt",
  "query_variants": [
    "Camera lia chậm trên mặt bàn làm việc, bắt đầu từ một bức ảnh/tranh một người đang bắt tay với Bác Hồ, sau đó chuyển sang phần bên phải là tay nghệ nhân đang vẽ lại chân dung người bắt tay Bác. Người nghệ nhân này vẽ bằng một cây bút đặc biệt trên một chất liệu đặc biệt",
    "Camera pans slowly across a desk, starting from a photo/painting of a person shaking hands with Uncle Ho (Ho Chi Minh), then shifting to the right showing an artisan's hand drawing a portrait of the person shaking hands. The artisan uses a specialized pen on a special material.",
    "camera lia mặt bàn làm việc bức ảnh bắt tay Bác Hồ tay nghệ nhân vẽ lại chân dung bút đặc biệt 

In [120]:
print(f"\n[*] Bắt đầu giải quyết query: {structured_query.get('query_id')}")
query_variants = structured_query.get("query_variants") or [structured_query.get("raw_query", "")]
entities = structured_query.get("entities", [])
raw_query = structured_query.get("raw_query", "")

# ---------------------------------------------------------
# LUỒNG 1: Visual Search (SigLIP / CLIP)
# ---------------------------------------------------------
print("[*] Chạy Luồng 1: Visual Search (SigLIP)...")
visual_multi = []
for q in query_variants:
    vec = encoder.encode_text(q) # Giả định encoder là SiglipEncoder
    visual_multi.append(retriever.search(vec, top_k=100)) # Giả định retriever là SiglipRetriever
visual_results = normalize_scores(reciprocal_rank_fusion(visual_multi))

# ---------------------------------------------------------
# LUỒNG 2: Lexical Text Search (BM25 - Từ khóa cứng)
# ---------------------------------------------------------
print("[*] Chạy Luồng 2: Lexical Text Search (BM25)...")
text_multi = [bm25_engine.search_bm25(q, top_k=100) for q in query_variants]
bm25_results = normalize_scores(reciprocal_rank_fusion(text_multi))

# ---------------------------------------------------------
# LUỒNG 3: Semantic Text Search (Caption Vector)
# ---------------------------------------------------------
print("[*] Chạy Luồng 3: Semantic Text Search (Caption Embedding)...")
semantic_multi = []
for q in query_variants:
    # Sử dụng caption_retriever tổng (chứa toàn bộ thư mục) mà bạn đã khởi tạo
    semantic_multi.append(caption_retriever.search(q, top_k=100))
semantic_results = normalize_scores(reciprocal_rank_fusion(semantic_multi))

# ---------------------------------------------------------
# DUNG HỢP 3 LUỒNG & BỘ LỌC VẬT THỂ
# ---------------------------------------------------------
fused_candidates = static_weight_fusion_3_way(
    visual_results, 
    bm25_results, 
    semantic_results, 
    w_visual=0.5, 
    w_bm25=0.2, 
    w_semantic=0.3
)

filtered_candidates = apply_object_filter(fused_candidates, entities, object_lookup)

print("\n=== TOP 50 KẾT QUẢ CHUNG CUỘC (SIGLIP + BM25 + SEMANTIC) ===")
for i, r in enumerate(filtered_candidates[:50], 1):
    print(f"{i:02d}. video={r['video_id']} frame={r['frame_id']} | Final={r['score']:.4f}")


[*] Bắt đầu giải quyết query: q_057
[*] Chạy Luồng 1: Visual Search (SigLIP)...
[*] Chạy Luồng 2: Lexical Text Search (BM25)...
[*] Chạy Luồng 3: Semantic Text Search (Caption Embedding)...
[Fusion] Trọng số: Visual=0.5, BM25=0.2, Semantic=0.3
[Object Score] Checking 2 object entities

=== TOP 50 KẾT QUẢ CHUNG CUỘC (SIGLIP + BM25 + SEMANTIC) ===
01. video=L30_V095 frame=549 | Final=0.8166
02. video=L30_V052 frame=539 | Final=0.5000
03. video=L30_V052 frame=563 | Final=0.4906
04. video=L22_V006 frame=31284 | Final=0.4656
05. video=L28_V004 frame=18035 | Final=0.4656
06. video=L30_V050 frame=2317 | Final=0.4418
07. video=L22_V001 frame=14847 | Final=0.4405
08. video=L28_V010 frame=12900 | Final=0.4399
09. video=L28_V004 frame=18507 | Final=0.4321
10. video=L28_V004 frame=18297 | Final=0.4254
11. video=L30_V052 frame=886 | Final=0.4236
12. video=L21_V014 frame=19983 | Final=0.4197
13. video=L30_V095 frame=409 | Final=0.4092
14. video=L30_V018 frame=6868 | Final=0.3795
15. video=L30_V052 

In [121]:

print("\n=== BẮT ĐẦU VLM RERANKING CHO TOP 20 ===")

top_k_rerank = 20
rerank_candidates = filtered_candidates[:top_k_rerank]
raw_query = structured_query.get("raw_query", "")

for candidate in rerank_candidates:
    vid = candidate["video_id"]
    fid = candidate["frame_id"]
    
    # 1. TÌM ĐƯỜNG DẪN ẢNH THỰC TẾ (Dùng Mapping)
    # Ánh xạ từ frame_id (VD: 6261) sang keyframe_n (VD: 125)
    n = object_lookup._get_keyframe_n(vid, fid)
    img_path = None
    
    if n is not None:
        img_dir = video_image_dir(vid)
        if os.path.exists(img_dir):
            # Quét tên file để xử lý mọi trường hợp padding (VD: 1.jpg, 001.jpg, 0001.jpg)
            for f_name in os.listdir(img_dir):
                if f_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    try:
                        # So sánh tên file (đã ép về số nguyên) với n
                        if int(os.path.splitext(f_name)[0]) == n:
                            img_path = os.path.join(img_dir, f_name)
                            break
                    except ValueError:
                        continue

    if not img_path or not os.path.exists(img_path):
        print(f" -> [Bỏ qua] Không tìm thấy file ảnh gốc cho {vid} | Frame: {fid} (n={n})")
        candidate["vlm_raw_score"] = None
        candidate["vlm_error"] = "image_not_found"
        continue
        
    # 2. CHẤM ĐIỂM BẰNG VLM
    print(f" -> Đang chấm điểm {vid} | Frame: {fid} (Đã tìm thấy file: {os.path.basename(img_path)})...")
    try:
        # Lấy caption hiện có làm ngữ cảnh phụ cho VLM
        caption_text = candidate.get("retrieval_text", "")
        vlm_score = vlm_engine.score(img_path, raw_query, caption=caption_text)
    except Exception as e:
        print(f"    [Lỗi VLM]: {e}")
        vlm_score = None
        candidate["vlm_error"] = str(e)

    candidate["vlm_raw_score"] = vlm_score

final_ranked, vlm_applied = finalize_vlm_ranking(
    filtered_candidates,
    top_k=top_k_rerank,
    vlm_weight=0.5,
)
if not vlm_applied:
    print("[VLM] Điểm không đủ độ phân tán; giữ nguyên retrieval ranking.")

print("\n=== KẾT QUẢ TOP 10 SAU KHI VLM RERANK ===")
for i, r in enumerate(final_ranked[:10], 1):
    vlm_display = r.get('vlm_raw_score')
    vlm_display = 'N/A' if vlm_display is None else f'{vlm_display}/10'
    print(f"{i:02d}. video={r['video_id']} frame={r['frame_id']} | Final={r['final_score']:.4f} (VLM={vlm_display} | Old={r['score']:.4f})")


=== BẮT ĐẦU VLM RERANKING CHO TOP 20 ===
 -> Đang chấm điểm L30_V095 | Frame: 549 (Đã tìm thấy file: 006.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L30_V052 | Frame: 539 (Đã tìm thấy file: 006.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L30_V052 | Frame: 563 (Đã tìm thấy file: 007.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L22_V006 | Frame: 31284 (Đã tìm thấy file: 269.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L28_V004 | Frame: 18035 (Đã tìm thấy file: 293.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L30_V050 | Frame: 2317 (Đã tìm thấy file: 029.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L22_V001 | Frame: 14847 (Đã tìm thấy file: 132.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L28_V010 | Frame: 12900 (Đã tìm thấy file: 270.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L28_V004 | Frame: 18507 (Đã tìm thấy file: 301.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L28_V004 | Frame: 18297 (Đã tìm thấy file: 297.jpg)...
[VLM RAW]: 0
 -> Đang chấm điểm L30_V052 | Frame: 886 (Đã tìm thấy file: 013.jpg)...
[VLM RAW]: 0
 -> Đang c

In [122]:
csv_output_dir = Path.home() / "Downloads"
if not csv_output_dir.is_dir():
    csv_output_dir = Path.home()

csv_output_path = csv_output_dir / "query-p2-36-kis.csv"

with csv_output_path.open(
    "w",
    encoding="utf-8",
    newline=""
 ) as csv_file:
    for result in final_ranked[:100]:
        csv_file.write(
            f"{result['video_id']}, {result['frame_id']}\n"
        )

